In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))

    # Real rule (verified in login.jsx): password must be at least 6 characters.
    # Error renders in p#password-error (verified in LoginInput.jsx).
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("123")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(2)

    err = wait.until(EC.visibility_of_element_located((By.ID, "password-error")))
    assert err.text.strip(), "Password error element is empty."
    print("Validation message:", err.text.strip())
    assert "6 characters" in err.text, f"Unexpected validation text: {err.text!r}"

    # Submission must be prevented: still on the login form, no session created
    assert driver.find_elements(By.ID, "username"), "Login form disappeared."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert not token, "Session was created despite invalid input."
    print("Submission prevented, no session created.")
    print("PASS: Form validation works")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("51_form_validation_FAIL.png")
finally:
    driver.quit()